# Capstone: publish and reconcile a batch table

*Author: [Sai Guntupalli](https://github.com/sai-guntupalli) · [sai-guntupalli/data-engineering-handbook](https://github.com/sai-guntupalli/data-engineering-handbook) · CC BY 4.0*

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sai-guntupalli/data-engineering-handbook/blob/master/notebooks/topics/concepts/batch-lakehouse-capstone.ipynb)

Follow a single delivery from landing through a replayable Delta write.

In [ ]:
# Setup: install notebook dependencies in Colab.
!pip install -q deltalake pyarrow~=26.0

## Setup and sample data

A delivery includes two orders and one replay. Amounts are integer cents.

In [ ]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pyarrow as pa
from deltalake import DeltaTable, write_deltalake

workspace = TemporaryDirectory()
table_path = str(Path(workspace.name) / "orders")
delivery = [(1, 1200), (2, 800), (1, 1200)]

## Prepare one row per business key

Validate repeated keys before producing a clean batch. A matching duplicate is a replay. Conflicting amounts need a source-version rule or quarantine.

**Where it matters**

Establish the table grain before publishing.

**Watch out for**

Dropping duplicate IDs silently can hide conflicting source records.

In [ ]:
clean = {}
for key, cents in delivery:
    if key in clean and clean[key] != cents:
        raise ValueError(f"Conflicting amounts for order {key}")
    clean[key] = cents
print("received:", len(delivery), "unique:", len(clean))

**What to notice:** Three arrivals produce two distinct orders.

## Publish and reconcile

Write the complete fixture atomically as a Delta table, replay the same write, and reconcile readback with the clean batch.

**Where it matters**

Separate retry safety from correctness of a particular commit.

**Watch out for**

The overwrite is valid because this input is a complete table snapshot. Partial deliveries need a bounded overwrite or merge.

In [ ]:
payload = pa.table({"id": list(clean), "cents": list(clean.values())})
for _ in range(2):
    write_deltalake(table_path, payload, mode="overwrite")
rows = DeltaTable(table_path).to_pyarrow_table().to_pylist()
print("rows:", len(rows), "expected cents:", sum(clean.values()), "published cents:", sum(r["cents"] for r in rows))

**What to notice:** The replay keeps two rows and the published total matches 2000 cents.

## Check yourself

### Exercise 1: Apply the pattern

**Difficulty:** Easy · about 5 min

How would you publish a delivery containing only a correction for order 2?

In [ ]:
# Try your solution here.

In [ ]:
print("Merge the changed row with a key and source-version rule.")

## Key takeaways

- Establish the table grain before publishing.
- Separate retry safety from correctness of a particular commit.

**Next:** [Capstone: Streaming orders](streaming-orders-capstone.ipynb)